In [12]:
# imports

import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI
from io import StringIO
import json
import pandas as pd
import subprocess
import sys

# If you get an error running this cell, then please head over to the troubleshooting notebook!

In [13]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


In [14]:
# Initialize OpenAI API client
openai = OpenAI()

In [15]:
from io import StringIO

import json
import pandas as pd
import subprocess
import sys

playwright_script = r"""
import json
from playwright.sync_api import sync_playwright

url = "https://www.flightaware.com/live/cancelled/"
with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    try:
        page = browser.new_page()
        page.goto(url, wait_until="domcontentloaded")
        page.locator("table").nth(3).wait_for()

        table_html = page.locator("table").evaluate_all(
            "(tables) => tables.slice(1, 4).map(table => table.outerHTML)"
        )
        print(json.dumps(table_html))
    finally:
        browser.close()
"""

result = subprocess.run(
    [sys.executable, "-c", playwright_script],
    check=True,
    stdout=subprocess.PIPE,
    text=True,
)
table_html = json.loads(result.stdout)

airline, origin, destination = [
    pd.read_html(StringIO(html))[0] for html in table_html
]

In [16]:
from IPython.display import Markdown, display

tables_for_summary = [
    ("By airline", airline),
    ("By origin airport", origin),
    ("By destination airport", destination),
]

table_text = "\n\n".join(
    f"{name} (first 10 rows):\n{frame.head(10).to_string(index=False)}"
    for name, frame in tables_for_summary
)

response = openai.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {
            "role": "system",
            "content": "Summarize these flight cancellation tables. Use only the supplied data; distinguish counts from percentages.",
        },
        {"role": "user", "content": table_text},
    ],
)

display(Markdown(response.choices[0].message.content))

Summary of flight cancellations and delays based on the supplied data:

By airline (top 10 airlines):
- Saudia had 264 flights cancelled (51%) and 35 delayed (6%).
- Flynas experienced 108 cancellations (43%) and 6 delays (2%).
- flyadeal had 57 cancellations (34%) and 6 delays (3%).
- Shenzhen Airlines had 36 cancellations (5%) but a higher number of delays at 73 (10%).
- IndiGo had 34 cancellations (1%) and 165 delays (7%).
- China Eastern saw 29 cancellations (1%) and 223 delays (10%).
- Batik Air reported 25 cancellations (8%) and 53 delays (17%).
- Qatar Airways had 25 cancellations (4%) and 33 delays (5%).
- Azul Brazilian Airlines showed 23 cancellations (2%) and 4 delays (0%).
- Akasa Air had 22 cancellations (10%) and 5 delays (2%).

By origin airport (top 10 airports):
- King Khaled Intl (RUH) had 257 cancellations (72%) and only 1 delay (0%).
- King Abdulaziz Intl (JED) recorded 72 cancellations (18%) and 30 delays (7%).
- King Fahd Intl (DMM) experienced 35 cancellations (29%) and 4 delays (3%).
- Soekarno-Hatta (CGK) had 21 cancellations (3%) and 69 delays (11%).
- Dubai Intl (DXB) saw 21 cancellations (4%) and 24 delays (4%).
- Cairo Intl (CAI) had 17 cancellations (5%) and 13 delays (4%).
- Prince Mohammad Bin Abdulaziz Intl (MED) reported 16 cancellations (18%) and 6 delays (6%).
- Hamad Intl (DOH) showed 15 cancellations (4%) and 17 delays (4%).
- Abha Rgnl (AHB) had 15 cancellations (36%) and 2 delays (4%).
- Chhatrapati Shivaji Maharaj Intl (BOM) showed 12 cancellations (2%) and 27 delays (6%).

By destination airport (top 10 airports):
- King Khaled Intl (RUH) had 255 cancellations (73%) and no delays (0%).
- King Abdulaziz Intl (JED) recorded 68 cancellations (18%) and 31 delays (8%).
- King Fahd Intl (DMM) had 34 cancellations (29%) and 7 delays (6%).
- Dubai Intl (DXB) showed 23 cancellations (4%) and 47 delays (9%).
- Cairo Intl (CAI) had 17 cancellations (5%) and 16 delays (5%).
- Prince Mohammad Bin Abdulaziz Intl (MED) experienced 15 cancellations (17%) and 7 delays (8%).
- Soekarno-Hatta (CGK) recorded 14 cancellations (2%) and a notable 78 delays (13%).
- Hamad Intl (DOH) had 14 cancellations (4%) and 22 delays (6%).
- Abha Rgnl (AHB) showed 14 cancellations (31%) and 2 delays (4%).
- Chhatrapati Shivaji Maharaj Intl (BOM) had 13 cancellations (3%) and 25 delays (5%).

Notably, King Khaled Intl (RUH) reflected the highest cancellation percentages both as origin and destination airport, with minimal delays. Saudia led airlines in cancellations with over half of its flights cancelled, while China Eastern and IndiGo showed higher delay counts.